In [ ]:
import numpy as np
from numba.core.types import none
from numba.np.arrayobj import np_arange

from main_functions import *
import matplotlib.pyplot as plt

recording_path = r"data\2026-02-25_mb_fish1_rec2"
save_path = r"results\2026-02-25_mb_fish1_rec2"


In [ ]:
fluorescence, recording, phase, ca_rec_group_id_fun = digest_folder(recording_path, plane=0)
process_recording(recording, phase, radial_bin_num=16)


In [ ]:
Dff_all_neuron = np.load(os.path.join(save_path, "deconvolved_Dff_original.npy"))
Dff_resampled = scipy.interpolate.interp1d(recording['ca_times'], Dff_all_neuron, kind='nearest')(
        recording['time_resampled'])

In [ ]:
# sample first neuron from loop (over all neurons)
i=0
test_neuron_dff = Dff_resampled[i, :]

# einfacher als mit GMMs (wie's in 2019 paper is), robust
detect_events_with_derivative(recording, test_neuron_dff)

In [ ]:
Dff_all_neuron.shape

# calculate reverse correlation function

In [ ]:
# ETA berechnung, Abb mit cnts für bewegungsrichtugen im vf
bootstrap_num=1000

# ROI data
test_neuron_signal_selection = recording['signal_selection'] # timepoints mask giving events
# Recording data
sample_rate = recording['sample_rate']
radial_bin_edges = recording['radial_bin_edges']
cmn_phase_selection = recording['cmn_phase_selection'] # timepoints mask giving CMN stim
motion_vectors_2d = recording['cmn_motion_vectors_2d']

motion_vectors_2d_filtered = motion_vectors_2d[test_neuron_signal_selection, :, :]
radial_bin_norms, radial_bin_etas = calculate_local_directions(motion_vectors_2d_filtered, radial_bin_edges)

recording[f'radial_bin_etas'] = radial_bin_etas # ETA's`
recording[f'radial_bin_norms'] = radial_bin_norms

min_frame_shift = 4 * sample_rate
max_frame_shift = int(cmn_phase_selection.sum() - min_frame_shift)
frame_shifts = np.random.randint(min_frame_shift, max_frame_shift, size=(bootstrap_num))

signal_within_cmn_selection = test_neuron_signal_selection[cmn_phase_selection]
signal_indices = signal_within_cmn_selection.nonzero()[0][:,None]


In [ ]:
idcs=np.mod(signal_indices + frame_shifts, signal_within_cmn_selection.size).T
idcs.shape, idcs[0].shape

In [ ]:
mv=motion_vectors_2d[cmn_phase_selection]

angles=np.arctan2(mv[:,:,0], mv[:,:,1])
velocities=np.linalg.norm(mv, axis=2)
mv.shape, angles.shape, velocities.shape

In [ ]:
vel=velocities[idcs[0]]
ang=angles[idcs[0]]
bins=radial_bin_edges
vel.shape, ang.shape, bins.shape

In [ ]:
bins[:-1].shape, ang[:, :, None].shape, vel[:, :, None].shape

In [ ]:
mv=mv[idcs[0]]

In [ ]:
# ang=np.arctan2(mv[:,:,0], mv[:,:,1])
# vel=np.linalg.norm(mv, axis=2)
# np.mean(vel[:, :, None] * np.logical_and(bins[:-1] <= ang[:, :, None], ang[:, :, None] <= bins[1:]), axis=0)

In [ ]:
norms = vel[:,  :, None] * np.logical_and(bins[:-1] <= ang[:,  :, None], ang[:, :, None] <= bins[1:])
etas=norms.mean(axis=0)

In [ ]:
etas.shape

In [ ]:
etas

In [ ]:
# radial_bin_bs_etas = np.zeros((bootstrap_num,) + radial_bin_etas.shape)
# bins = radial_bin_bs_etas
# for s in tqdm(range(bootstrap_num)):
#         # Circularly permutate signal
#         ### perm_signal_selection = np.roll(signal_within_cmn_selection, frame_shifts[s])
#
#         # Get motion vectors of permutated signal
#         bs_motion_vectors = motion_vectors_2d[cmn_phase_selection][perm_signal_selection]
#
#         # Calculate vector ETAs for each local radial bin
          # REPLACED BELOW
#         #radial_bin_bs_etas[s] = calculate_local_directions(bs_motion_vectors, radial_bin_edges)[1]
#
#         motion_angles = np.arctan2(bs_motion_vectors[:, :, 1], bs_motion_vectors[:, :, 0])
#         motion_velocities = np.linalg.norm(bs_motion_vectors, axis=2)
#         from numpy import newaxis
#         bin_norms = motion_velocities[:, :, newaxis] * np.logical_and(radial_bin_edges[:-1] <= motion_angles[:, :, newaxis], motion_angles[:, :, newaxis] <= radial_bin_edges[1:])
#         radial_bin_bs_etas[s] = np.mean(bin_norms, axis=0)
#
# recording[f'radial_bin_bs_etas'] = radial_bin_bs_etas

### functions

In [ ]:
# def calculate_local_directions(motvecs: np.ndarray, bin_edges: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
#     # Convert to angle and velocity
#     motion_angles = np.arctan2(motvecs[:, :, 1], motvecs[:, :, 0])
#     motion_velocities = np.linalg.norm(motvecs, axis=2)
#     # Backwards transform (keep for reference):
#     # motion_vectors_2d = np.zeros((*motion_angles.shape[:2], 2))
#     # motion_vectors_2d[:,:,0] = motion_velocities * np.cos(motion_angles)
#     # motion_vectors_2d[:,:,1] = motion_velocities * np.sin(motion_angles)
#
#     # Calculate bin vectors for each patch and frame weighted by the local velocities
#     bin_norms = motion_velocities[:, :, None] * np.logical_and(bin_edges[:-1] <= motion_angles[:, :, None],
#                                                                motion_angles[:, :, None] <= bin_edges[1:])
#
#     # Calculate ETAs
#     bin_etas = np.mean(bin_norms, axis=0)
#
#     return bin_norms, bin_etas

# compare results to original pipeline results

In [ ]:
import pickle

In [ ]:
with open('radial_bin_etas.npy', 'rb') as f:
    radial_bin_etas = np.load(f)


with open('radial_bin_etas_slow.pkl', 'rb') as f:
    radial_bin_etas_slow = pickle.load(f)

etas, etas_ = radial_bin_etas, radial_bin_etas_slow
etas.shape, etas_.shape

In [ ]:
error=np.sum(radial_bin_etas - radial_bin_etas_slow)
error/ radial_bin_etas.sum() , error/radial_bin_etas_slow.sum()

# run full pipeline for one neuron

In [ ]:
# einfacher als mit GMMs (wie's in 2019 paper is), robust
detect_events_with_derivative(recording, test_neuron_dff)

# Calcium-event-triggered-averages berechnung
calculate_reverse_correlations_shm(recording, bootstrap_num=1024)

# 1. Teil Bootstrap test, bootstrap verteilung berechnungen, welche revcorr significant
calculate_directional_significance(recording)

# added by me: 2. Teil
calculate_directional_preference(recording)

# clusteranalyse
find_clusters(recording)

# cluster based statistics, (2-step NP bootsrap test)
calculate_cluster_significances(recording)

plot_rf_overview(recording, i, save_path)

In [ ]:
f'test{str(1)}'

# inspect recording dict

In [ ]:
save_path

In [ ]:
import pickle
with open(os.path.join(recording_path, 'recording_dicts', f'recording_neuron{str(i)}.pkl'), 'rb') as f:
    recording = pickle.load(f)

In [ ]:
recording.keys()

In [ ]:
recording['radial_bin_bs_etas'].shape, recording['radial_bin_bs_etas'].shape

In [ ]:
p=recording['radial_bin_p_values']

In [ ]:
plt.hist(p.flatten(), bins=200);
np.sum(p<.05), p.size

In [ ]:
np.any(p>.05, axis=1).sum()

In [ ]:
recording['radial_bin_significances']

# fitting optic flow field

scipy minimize, dont care about values, since only angles count.
define rotation axis: azimuth angle, elevation angle



In [ ]:
radial_bin_centers=recording['radial_bin_centers']
radial_bin_centers

In [ ]:
recording['radial_bin_bs_etas'].nbytes/1024**2

In [ ]:
positions=recording['positions']
positions.shape

In [ ]:
fig=plt.figure()
ax=fig.add_subplot(projection='3d')
ax.scatter(recording['positions'][:,0],recording['positions'][:,1],recording['positions'][:,2])
ax.scatter(0,0,0)
plt.show()

In [ ]:
import scipy

In [ ]:
def TOF(angle_azimuth, angle_elevation):
    """
        Translational optic flow field.
        Returns a translational optic flow field for a given translation axis.

        Parameters:
            angle_azimuth (float):
                angle in radians of the azimuth of the translation axis.
            angle_elevation (float):
                angle in radians of the elevation of the translation axis.
            positions (np.array):
                3D positions to sample flow field at.

        Returns:
            F (np.array):
                Return values of optic flow field (3D vector) at each point
                given in positions.
            FoC, FoE (np.array):
                Focus of Expansion and Focus of Contraction of the optic flow field.
                Given as 3D unit vector.
    """


In [ ]:
positions.shape

In [ ]:
def RSSangle(F, E):
